# Module 5, Topic 2 — Implementing a Basic Word Embedding Model (Word2Vec)

**Generative AI Fellowship — Beginner**

In this notebook, we will train a small Word2Vec model **from scratch** on a tiny Nigerian-English corpus, and inspect what it learns.

Each cell introduces exactly **one new concept** — run them in order, top to bottom.

**What we'll do:**
1. Install and import what we need
2. Look at our training corpus
3. Train a Word2Vec model on it
4. Look up a word's vector
5. Find a word's nearest neighbours
6. Try vector arithmetic
7. See what happens with a word we never trained on

> 💡 This is a *toy* corpus (a few dozen sentences) built to make the ideas visible quickly. Real Word2Vec models are trained on millions of sentences — with a corpus this small, the neighbours we find will be a rough sketch, not a polished result. That's expected, and we'll call it out as we go.

## Step 1 — Install gensim

We'll use `gensim`, a popular Python library with a built-in Word2Vec implementation. This means we don't have to write the training loop from scratch by hand — we focus on understanding what's happening conceptually.

In [ ]:
!pip install gensim --quiet

## Step 2 — Import what we need

`Word2Vec` is the model class we'll train. `pprint` just makes printed output easier to read.

In [ ]:
from gensim.models import Word2Vec
from pprint import pprint

print("Ready to go!")

## Step 3 — Our training corpus

Word2Vec learns entirely from example sentences — no dictionary, no manual labels.

Below is a small set of Nigerian-English sentences about markets, transport, and food. Notice how words like `trader`, `market`, and `price` appear together a lot, and separately, words like `danfo`, `bus`, and `okada` appear together a lot too. That repeated pattern is exactly what Word2Vec will pick up on.

Each sentence is already split into a list of lowercase words — this is the format gensim's Word2Vec expects.

In [ ]:
corpus = [
    "the trader sold garri at the mile 12 market".split(),
    "prices at the market went up this week".split(),
    "i bought fabric from a trader in balogun market".split(),
    "the market was full of traders selling rice and beans".split(),
    "every trader at the market complained about high prices".split(),
    "she is a trader who sells tomatoes at the local market".split(),
    "the price of garri increased at the market again".split(),
    "traders at the market negotiate prices with customers".split(),
    "i boarded the danfo to yaba this morning".split(),
    "the danfo conductor collected fare from every passenger".split(),
    "i boarded the bus to yaba this morning".split(),
    "the bus conductor collected fare from every passenger".split(),
    "an okada is faster than a bus in lagos traffic".split(),
    "i took an okada because the danfo was too slow".split(),
    "the keke napep dropped me close to the market".split(),
    "danfo bus and okada are common transport options in lagos".split(),
    "traffic in lagos makes the bus ride very slow".split(),
    "the okada rider knows all the fastest routes in lagos".split(),
    "jollof rice is a popular dish at every party in lagos".split(),
    "she cooked jollof rice and chicken for the party".split(),
    "suya is a popular snack sold at night in lagos".split(),
    "we ate suya and jollof rice at the party".split(),
    "the aroma of jollof rice filled the whole street".split(),
    "akara and pap is a common breakfast in lagos".split(),
    "moin moin and jollof rice were served at the wedding".split(),
    "the chef prepared jollof rice suya and moin moin".split(),
    "she paid for the jollof rice with her card".split(),
    "the trader gave change in naira after the sale".split(),
    "he transferred naira to pay for the fabric".split(),
    "the price was quoted in naira at the market".split(),
    "she withdrew naira from the atm near the market".split(),
    "the bank charged a fee for the naira transfer".split(),
]

print(f"Corpus size: {len(corpus)} sentences")
print("First sentence:", corpus[0])

## Step 4 — Train the Word2Vec model

A few parameters worth knowing:

- `vector_size` — how many numbers make up each word's vector (embedding size)
- `window` — how many neighbouring words count as "context" (see Topic 2 slides)
- `min_count` — ignore words that appear fewer than this many times (we set it to 1 since our corpus is tiny)
- `sg` — training setup: `1` = skip-gram, `0` = CBOW
- `epochs` — how many passes to make over the corpus during training
- `seed` — makes the random starting vectors reproducible, so you get the same results each run

This single line of code is running the entire predict → compare → adjust loop from the slides, thousands of times, behind the scenes.

In [ ]:
model = Word2Vec(
    sentences=corpus,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    epochs=200,
    seed=42,
    workers=1,
)

print("Vocabulary size:", len(model.wv.key_to_index))
print("Vector size per word:", model.wv.vector_size)

## Step 5 — Look up a word's vector

After training, the model is really just a lookup table: word in, vector out.

We'll only print the first 10 numbers of the vector — the real vector has 50 numbers, which is too many to read comfortably.

In [ ]:
market_vector = model.wv["market"]

print("Shape of the vector:", market_vector.shape)
print("First 10 numbers:")
print(market_vector[:10])

## Step 6 — Find the nearest neighbours of "market"

This is the real test: did the model learn anything meaningful?

`most_similar` returns the words whose vectors are closest to the given word's vector, along with a cosine similarity score (closer to 1.0 = more similar). We'll cover exactly what cosine similarity means in Topic 3 — for now, just read it as "how close."</br>

In [ ]:
pprint(model.wv.most_similar("market", topn=5))

**What to look for:** words like `trader`, `price`, or `traders` should be showing up near the top — words that appeared in similar sentence contexts to "market" throughout our corpus.

If the ranking looks a little rough — that's expected. Real Word2Vec models are trained on **millions** of sentences; ours only saw a few dozen. The pattern should still be visible, just noisier than a production model.

## Step 7 — Try it with a transport word

Let's check whether "danfo" landed near other transport words, the way we'd expect from Slide 8 of the Topic 2 deck.

In [ ]:
pprint(model.wv.most_similar("danfo", topn=5))

You should see transport-related words like `bus`, `okada`, or `conductor` ranking higher than food or money words like `jollof` or `naira`.

## Step 8 — Measure similarity between two specific words directly

Instead of asking "what's closest to this word?", we can also directly ask "how similar are these two specific words?" — this returns one number between -1 and 1.

In [ ]:
print("similarity(danfo, bus)         =", model.wv.similarity("danfo", "bus"))
print("similarity(danfo, jollof)        =", model.wv.similarity("danfo", "jollof"))
print("similarity(market, trader)       =", model.wv.similarity("market", "trader"))

Compare the three numbers above. If training worked as expected, `danfo`–`bus` and `market`–`trader` should score noticeably higher than `danfo`–`jollof`.

## Step 9 — Vector arithmetic (bonus)

Recall from the slides: `king - man + woman ≈ queen`. Let's try a small-scale version of that same idea with our own corpus.

`most_similar` supports this directly with `positive` and `negative` word lists — it adds and subtracts the vectors for you, then finds the closest word to the result.

In [ ]:
pprint(
    model.wv.most_similar(positive=["market", "danfo"], negative=["trader"], topn=5)
)

With a corpus this small, don't expect a clean, obviously "correct" answer the way the king/queen example works on large-scale models — but you should see the results are not random either. Try swapping in different words above and see what comes out.

## Step 10 — What happens with a word we never trained on?

Word2Vec has a well-known limitation: it can only produce a vector for words it saw during training. Let's see what happens when we ask for a word that never appeared in our corpus.

In [ ]:
try:
    model.wv["airplane"]
except KeyError as e:
    print("Error:", e)

This is the **out-of-vocabulary (OOV)** problem mentioned on Slide 12 of the Topic 2 deck. It's one of the reasons production systems often use pretrained embedding models trained on huge, general-purpose corpora, or newer embedding techniques that handle unseen words more gracefully.

## Recap

In this notebook, we:
- Trained a Word2Vec model from scratch on a small Nigerian-English corpus
- Looked up a word's raw vector
- Found nearest neighbours for `market` and `danfo`, and saw them cluster sensibly
- Measured direct similarity between specific word pairs
- Tried simple vector arithmetic
- Hit the out-of-vocabulary limitation firsthand

**Up next (Topic 3):** we'll formalise exactly *how* "closeness" is measured — cosine similarity, Euclidean distance — and see why brute-force nearest-neighbour search like `most_similar` above doesn't scale once we have thousands or millions of documents.